# Davies-Bouldin Index (DBI) Implementation

The **Davies-Bouldin Index** is an internal evaluation metric used to validate clustering algorithms (like GMM and K-Means) without requiring ground truth labels. It measures the average "similarity" between clusters, where similarity is a function that compares the distance within a cluster to the distance between clusters.

***Interpretation:** A **lower** DB score indicates better clustering (clusters are compact and well-separated).*

### 1. Mathematical Definition

The index is defined as the average similarity measure of each cluster with its most similar cluster:
$$DB = \frac{1}{K} \sum_{i=1}^{K} \max_{j \neq i} (R_{ij})$$

Where:

* $K$: The number of clusters.
* $R_{ij}$: The similarity ratio between cluster $i$ and cluster $j$ .

The similarity ratio $R_{ij}$ is calculated as:
$$R_{ij} = \frac{s_i + s_j}{d_{ij}}$$

### 2. Component Derivations

#### A. Intra-Cluster Scatter ($s_i$)

This measures the **compactness** of a cluster. It is the average Euclidean distance between every point in cluster $i$ and its centroid $\mu_i$.
$$s_i = \frac{1}{|C_i|} \sum_{x \in C_i} \| x - \mu_i \|_2$$

* **In Code:** We compute this using `np.mean` of the Euclidean distances (`np.sqrt(np.sum(...))`) for all points belonging to a specific label.

#### B. Inter-Cluster Separation ($d_{ij}$)

This measures the **separation** between two clusters. It is the Euclidean distance between the centroids of cluster  $i$  and cluster $j$.
$$d_{ij} = \| \mu_i - \mu_j \|_2$$

* **In Code:** Calculated as the norm of the difference vector between two centroids: `np.sqrt(np.sum((centroids[i] - centroids[j])**2))`.

#### C. The Ratio ($R_{ij}$)
$$R_{ij} = \frac{\text{Compactness}_i + \text{Compactness}_j}{\text{Separation}_{ij}}$$

* **Logic:**
* If clusters are tight ($s$ is low) and far apart ($d$ is high), the ratio is **small**.
* If clusters are loose ($s$ is high) or overlapping ($d$ is low), the ratio is **large**.
* We iterate through all pairs and find the **worst-case** (maximum) ratio for each cluster.



### 3. Implementation Details

Our implementation adheres to the following logic:

1. **Centroids:** Compute the mean vector for every unique cluster label found in the data.
2. **Scatter ($s_i$):** Calculate the mean distance of points from their respective centroids.
3. **Pairwise Comparison:** For every cluster $i$, compare it against every other cluster $j$ to calculate $R_{ij}$.
4. **Max Selection:** Find the maximum $R_{ij}$ (the "worst neighbor") for cluster $i$.
5. **Averaging:** The final score is the average of these maximums.

* **Edge Case Handling:**
* If `n_clusters < 2`, the score is undefined (returns `np.inf`).
* If two centroids overlap exactly ($d_{ij} = 0$), the ratio is treated as `np.inf` to penalize the model heavily.

# Calinski-Harabasz Index

The index works by comparing two types of dispersion (the sum of squared distances):

     Between-Cluster Dispersion (SSB): Measures how far apart the different clusters are from each other.

     Within-Cluster Dispersion (SSW): Measures how compact or "tight" the data points are within their own cluster.

The index is calculated as the ratio of these two values, normalized by the number of clusters ($k$) and the total number of data points ($N$):

$$CH = \frac{SSB/(k−1)​}{SSW/(N−k)}$$

A high score means the clusters are both dense (points are close together inside the cluster) and well-separated (clusters are far apart from each other).